# 🔬 AirSense — Exploratory Data Analysis (EDA)
**MSc Business Intelligence & Analytics | Capstone Project (Week 4)**

This notebook conducts in-depth statistical exploratory data analysis on real-time air quality telemetry stored in TimescaleDB.

### Core Analytical Questions:
1. **City Distributions**: How do AQI, PM2.5, and PM10 vary across Ahmedabad, Bengaluru, Pune, Gandhinagar, Delhi, and Mumbai?
2. **Diurnal Rhythms**: Are there distinct rush-hour morning/evening spikes compared to midday solar dissipation?
3. **Particulate Diagnostics**: What is the PM2.5 to PM10 ratio across regions (fine combustion particles vs. coarse mechanical dust)?
4. **Outlier Detection**: Which readings exhibit abnormal standard deviations ($Z > 2.0$) indicating localized pollution surges?

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sqlalchemy import text

# Ensure project root is available
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from common.config import load_settings
from consumer.db import create_db_engine

settings = load_settings(PROJECT_ROOT / "config.yaml")
engine = create_db_engine(settings)
print("✅ Connected to TimescaleDB successfully!")

In [ ]:
# Load historical telemetry directly from TimescaleDB hypertable
query = text("""
    SELECT time, city, city_display_name, aqi, pm25, pm10, no2, co, so2, o3, fetched_at
    FROM aqi_readings
    ORDER BY time ASC;
""")

with engine.connect() as conn:
    df = pd.read_sql(query, conn)

print(f"Loaded {len(df)} telemetry records across {df['city'].nunique()} cities.")
df.head()

## 1. Summary Statistics & Cross-City Distribution

In [ ]:
stats = df.groupby('city_display_name')[['aqi', 'pm25', 'pm10']].agg(['count', 'mean', 'min', 'max'])
stats.round(1)

## 2. Diurnal (Hourly) Rhythm Modeling
Analyzing air quality fluctuations by hour of the day.

In [ ]:
df['time'] = pd.to_datetime(df['time'])
df['hour'] = df['time'].dt.hour
diurnal = df.groupby('hour')['aqi'].mean()
print('Hourly Average AQI:')
print(diurnal)

## 3. Particulate Diagnostic (PM2.5 / PM10 Ratio)
- Ratio > 0.6 indicates fine combustion soot & tailpipe emissions.
- Ratio < 0.4 indicates coarse mechanical/road dust.

In [ ]:
df['pm_ratio'] = df['pm25'] / df['pm10']
ratio_by_city = df.groupby('city_display_name')['pm_ratio'].mean()
print('Average PM2.5 / PM10 Ratio:')
print(ratio_by_city.round(2))

## 4. Statistical Anomaly & Spike Detection (Z-Score)
Detects readings deviating significantly ($Z \ge 1.5$) from the historical mean.

In [ ]:
df['z_score'] = (df['aqi'] - df['aqi'].mean()) / (df['aqi'].std() if df['aqi'].std() > 0 else 1)
spikes = df[df['z_score'].abs() >= 1.5]
print(f"Detected {len(spikes)} statistical spikes (Z >= 1.5):")
spikes[['time', 'city_display_name', 'aqi', 'z_score']]